# HateGuard 

Toxic Comment Detection
and Video Analysis

In [ ]:
import os
# 1. cloning the repository
!rm -rf hate-comment-dectection
!git clone https://github.com/ATANU28-bit/hate-comment-dectection.git
%cd hate-comment-dectection

In [ ]:
# 2. Installing dependencies
!sudo apt update && sudo apt install ffmpeg -y
!pip install -r requirements.txt

# Upgrade libraries to latest versions
!pip install --upgrade youtube-comment-downloader pytubefix uvicorn nest_asyncio

# Installing UI dependencies
%cd ui
!npm install
%cd ..

### 3. One-Time YouTube Authentication
Run the cell below. It will show a link and a code. 
1. Click the link (google.com/device).
2. Enter the code shown in the output.
3. This authorizes audio downloads so you are not detected as a bot.

In [ ]:
from pytubefix import YouTube
import os
print("Starting Authentication flow...")
try:
    # We must try to access streams to force the OAuth prompt and CACHE the token
    yt = YouTube('https://youtube.com/watch?v=jNQXAC9IVRw', use_oauth=True, allow_oauth_cache=True)
    _ = yt.streams.first() # This line triggers the actual login request
    print("\nSUCCESS: Login cached! You can now start the server.")
except Exception as e:
    print(f"Authentication error: {e}")

In [ ]:
import uvicorn
import nest_asyncio
import threading
import time
import sys
import subprocess
from src.api import app
from google.colab.output import eval_js

# Allow uvicorn to run inside the notebook
nest_asyncio.apply()

# 1. Start Backend in a Thread (Sharing memory for OAuth tokens)
def run_backend():
    print("Starting Backend Server...")
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="info")

threading.Thread(target=run_backend, daemon=True).start()

# 2. Configure UI env
with open("ui/.env", "w", encoding="utf-8") as f:
    f.write("VITE_API_URL=/api\n")

# 3. Start Frontend via Subprocess (Vite needs Node.js)
print("Starting Frontend Vite proxy...")
frontend = subprocess.Popen(
    ["npm", "run", "dev", "--prefix", "ui", "--", "--host", "127.0.0.1", "--port", "5173"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

def stream_logs(proc):
    for line in iter(proc.stdout.readline, b''):
        sys.stdout.write(line.decode('utf-8'))
        sys.stdout.flush()

threading.Thread(target=stream_logs, args=(frontend,), daemon=True).start()

time.sleep(5)

proxy_url = eval_js("google.colab.kernel.proxyPort(5173)")
print(f"\n TO OPEN APP: {proxy_url}\n")

try:
    frontend.wait()
except KeyboardInterrupt:
    print("Shutting down...")
    frontend.terminate()